# A Simple Compiler for a Fragment of  `C`

This file shows how a simple compiler for a fragment of the programming language `C` can be implemented using `FsLexYacc`.

First, we describe the grammar of our language using EBNF-notation, since this enables us to give a very concise presentation of the grammar.
Below, the token 'NUMBER' stands for natural numbers and 'ID' represents identifiers.
```
program: function+
       
function: 'int' ID '(' ('int', ID (',' 'int' ID)*)?  ')' '{' decl* stmnt* '}' 
    
decl: 'int' ID ';'  
    
stmnt: '{' stmnt* '}'
     | ID '=' expr ';' 
     | 'if' '(' boolExpr ')' stmnt 
     | 'if' '(' boolExpr ')' stmnt 'else' stmnt 
     | 'while' '(' boolExpr ')' stmnt 
     | 'return' expr ';'
     | expr ';'             
     
boolExpr: expr ('==' | '!=' | '<=' | '>=' | '<' | '>')  expr 
        | '!' boolExpr             
        | boolExpr ('&&' | '||') boolExpr
        | '(' boolExpr ')'
     
expr: expr ('+' | '-' | '*' | '/' | '%') expr 
    | '(' expr ')' 
    | NUMBER
    | ID ('(' (expr (',' expr)*)? ')')?  
```
The FsYacc specification below stratifies operator precedence and resolves the dangling `else` explicitly. It additionally implements the bundled exercises: C-style `for`, range `for`, unary minus and the ternary conditional. Both loops desugar to `Assign`/`While`/`Block` AST nodes. 

## Specification of the Scanner

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "nuget: FsLexYacc.Runtime, 11.3.0"
#r "IntegerC/bin/Debug/net9.0/IntegerC.dll"
open System
open System.IO
open IntegerC
// Build IntegerC/IntegerC.fsproj before running this notebook.

In [ ]:
let astTypes = File.ReadAllText("IntegerC/Ast.fs")
printfn "%s" astTypes

The token `NUMBER` specifies a natural number.

In [ ]:
let numberRule = """| '0' | ['1'-'9'] digit* { NUMBER(int (text lexbuf)) }"""
printfn "%s" numberRule

Below, we define the tokens for operator symbols consisting of more than one character.

In [ ]:
let operatorRule = """| "==" { EQ } | "!=" { NE } | "<=" { LE } | ">=" { GE }"""
printfn "%s" operatorRule

Integer C allows line and block comments. FsLex uses a separate recursive `comment` rule that stops at the first `*/` and updates source positions for each newline.

In [ ]:
let commentRules = """| "/*" { comment lexbuf; token lexbuf }
and comment = parse
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; comment lexbuf }
| eof { failwith "Unclosed block comment" }
| _ { comment lexbuf }"""
printfn "%s" commentRules

The keywords `'int', 'if', 'else', 'while', 'return'` have to be dealt with separately as they are syntactical identical to identifiers.  The dictionary `Keywords` shown below maps every keyword to its *token type*.

In [ ]:
let keywordRule = """    | "int" -> INT | "if" -> IF | "else" -> ELSE | "while" -> WHILE | "for" -> FOR | "return" -> RETURN"""
printfn "%s" keywordRule

When an identifier is read, we first have to check whether the identifier is one of our keywords.  If so, we assign the corresponding token type that is stored in the dictionary `Keywords`.  Otherwise, the token type is set to `ID`.

In [ ]:
let identifierRule = """let identifier = ['a'-'z' 'A'-'Z'] ['a'-'z' 'A'-'Z' '0'-'9' '_']*
| identifier { keyword (text lexbuf) }"""
printfn "%s" identifierRule

Operators consisting of a single character do not need an associated token type.

In [ ]:
let literalRules = """| '+' { PLUS } | '-' { MINUS } | '*' { TIMES } | '/' { DIVIDE } | '%' { MOD }"""
printfn "%s" literalRules

White space, i.e. *space characters*, *tabulators*, and *carriage returns* are ignored. 

In [ ]:
let whitespaceRule = """| [' ' '\t' '\r']+ { token lexbuf }"""
printfn "%s" whitespaceRule

Syntactically, newline characters are ignored. However, we still need to keep track of them in order to know the current line number, which is used for error messages.

In [ ]:
let newlineRule = """| "//" [^ '\n']* { token lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; comment lexbuf }"""
printfn "%s" newlineRule

`LexBuffer<char>.StartPos` contains the line and column of the token. Columns are converted to one-based values for diagnostics.

In [ ]:
// LexBuffer.StartPos.Column + 1 is the human-readable column.
let position source = Frontend.scan source |> List.map (fun (_,line,column) -> line,column)

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
let errorRule = """| _ { failwithf "Illegal character %s at line %d, column %d" (text lexbuf) lexbuf.StartPos.Line (lexbuf.StartPos.Column+1) }"""
printfn "%s" errorRule

Build-time scanner generation needs no notebook filename workaround.

In [ ]:
let scanner = Frontend.scan

In [ ]:
let testScanner fileName =
    let source = File.ReadAllText fileName
    printfn "%s" source
    scanner source

In [ ]:
File.ReadAllText("Examples/MySum.c") |> printfn "%s"

In [ ]:
for token in testScanner "Examples/MySum.c" do printfn "%A" token

## Specification of the Parser

FsYacc uses BNF productions with typed semantic actions. The generated parser produces `Function list`; statements and expressions are discriminated unions defined in `IntegerC/Ast.fs`.

The *start variable* of our grammar is `program`.

In [ ]:
let startDeclaration = """%start program"""
printfn "%s" startDeclaration

The `IF` and `ELSE` precedence declarations attach each `else` to the nearest unmatched `if`. Arithmetic, Boolean and ternary precedence are expressed with layered grammar variables. The order from weakest to strongest is ternary `?:`, `||`, `&&`, Boolean `!`, comparisons, addition/subtraction, multiplication/division/remainder and unary minus.

The original lecture grammar applies `!` to a Boolean expression: `!x < y` means `!(x < y)`. The `logicalNot` layer preserves that behavior. Parentheses make the intended Boolean operand explicit; the short-circuit rules for `&&` and `||` are unchanged.


In [ ]:
let danglingElsePrecedence = """%nonassoc IF
%nonassoc ELSE"""
printfn "%s" danglingElsePrecedence

In [ ]:
let programRules = """functions: functionDef { [$1] } | functionDef functions { $1::$2 }"""
printfn "%s" programRules

In [ ]:
let functionRule = """functionDef: INT ID LPAREN parameters RPAREN LBRACE declarations statements RBRACE"""
printfn "%s" functionRule

In [ ]:
let parameterRules = """parameters: { [] } | INT ID { [$2] } | INT ID COMMA parameters { $2::$4 }"""
printfn "%s" parameterRules

In [ ]:
let declarationRules = """declarations: { [] } | INT ID SEMICOLON declarations { $2::$4 }"""
printfn "%s" declarationRules

In [ ]:
let statementListRules = """statements: { [] } | statement statements { $1::$2 }"""
printfn "%s" statementListRules

The `if` production without `else` has `%prec IF`, while the longer production has `ELSE` precedence.

In [ ]:
let parserSpecification = File.ReadAllText("IntegerC/Parser.fsy")
printfn "%s" parserSpecification

In [ ]:
let comparisonRules = """logicalAnd: logicalNot { $1 } | logicalAnd AND logicalNot { Binary("&&",$1,$3) }
logicalNot: comparison { $1 } | NOT logicalNot { Not $2 }
comparison: arithmetic { $1 } | arithmetic LT arithmetic { Binary("<",$1,$3) }"""
printfn "%s" comparisonRules

In [ ]:
let arithmeticRules = """arithmetic:"""
printfn "%s" arithmeticRules

In [ ]:
let argumentRules = """arguments: { [] } | nonemptyArguments { $1 }"""
printfn "%s" argumentRules

In [ ]:
// Frontend.parse wraps the generated parser with line/column diagnostics.

FsYacc generates and compiles parser tables during the project build. No runtime table-writing options are needed.

In [ ]:
let parser = Frontend.parse

The layered expression grammar and explicit dangling-else precedence yield zero unresolved parser conflicts.

In [ ]:
// The build reports zero unresolved parser conflicts.

The typed AST is printed structurally and converted to Graphviz DOT by the recursive functions below. Open the saved DOT file in Graphviz, or render it to SVG with `dot -Tsvg`.

In [ ]:
let astToDot functions =
    let output = System.Text.StringBuilder("digraph AST {\n  node [shape=box];\n")
    let mutable nextId = 0
    let node (title: string) (children: string list) =
        let id = "n" + string nextId
        nextId <- nextId + 1
        let escaped = title.Replace("\\", "\\\\").Replace("\"", "\\\"")
        output.AppendLine(sprintf "  %s [label=\"%s\"];" id escaped) |> ignore
        for child in children do output.AppendLine(sprintf "  %s -> %s;" id child) |> ignore
        id
    let rec expr = function
        | Number n -> node (string n) []
        | Variable name -> node name []
        | Binary(op,lhs,rhs) -> node op [expr lhs; expr rhs]
        | Not value -> node "!" [expr value]
        | Call(name,args) -> node ("call " + name) (List.map expr args)
        | Conditional(test,yes,no) -> node "?:" [expr test; expr yes; expr no]
    let rec statement = function
        | Assign(name,value) -> node (name + " =") [expr value]
        | If(test,yes,no) -> node "if" ([expr test; statement yes] @ (no |> Option.map statement |> Option.toList))
        | While(test,body) -> node "while" [expr test; statement body]
        | Return value -> node "return" [expr value]
        | Block statements -> node "block" (List.map statement statements)
        | Expression value -> node "expression" [expr value]
    functions
    |> List.map (fun f -> node ("int " + f.Name + "(" + String.concat ", " f.Parameters + ")") (List.map statement f.Body))
    |> node "program" |> ignore
    output.AppendLine("}").ToString()


`parse fileName` reads a C source file and returns its typed AST. The frontend starts a fresh scanner buffer for each call and reports source positions on errors.

In [ ]:
let parse fileName =
    let ast = parser (File.ReadAllText fileName)
    printfn "%A" ast
    ast

In [ ]:
File.ReadAllLines("Examples/MySum.c") |> Array.iteri (fun i line -> printfn "%3d %s" (i+1) line)

In [ ]:
let sumAst = parse "Examples/MySum.c"
let dot = astToDot sumAst
File.WriteAllText("Examples/MySum.dot",dot)
printfn "%s" dot
// Optional rendering: dot -Tsvg Examples/MySum.dot -o Examples/MySum.svg

The function `indent` is used to indent the generated assembler commands by preceding them with 8 space characters. 

In [ ]:
let indent command = "        " + command
let label name = "    " + name + ":"
let createLabels () =
    let mutable count = 0
    fun () -> count <- count + 1; "l" + string count

type CompilationContext = {
    Symbols: Map<string,int>
    Arities: Map<string,int>
    ClassName: string
    NewLabel: unit -> string
    ReturnsVoid: bool
}

`compileExpr`, `compileBool` and `compileStatement` receive a `CompilationContext` and a typed AST. The context holds local-variable slots, function arities, class name and a label factory. Each returns `(instructions, maximumStack)`, so the JVM stack limit is computed while generating code. Arithmetic operands contribute different temporary stack heights; branch arms meet at equal heights.

In [ ]:
let rec compileExpr context expression : string list * int =
    let command text = indent text
    let binary instruction lhs rhs =
        let left, leftSize = compileExpr context lhs
        let right, rightSize = compileExpr context rhs
        left @ right @ [command instruction], max leftSize (1 + rightSize)
    match expression with
    | Number number -> [command ("ldc " + string number)], 1
    | Variable name ->
        match Map.tryFind name context.Symbols with
        | Some slot -> [command ("iload " + string slot)], 1
        | None -> failwithf "Undeclared variable: %s" name
    | Binary (("+" | "-" | "*" | "/" | "%") as op,lhs,rhs) ->
        let instruction = Map.ofList ["+","iadd"; "-","isub"; "*","imul"; "/","idiv"; "%","irem"]
        binary instruction.[op] lhs rhs
    | Binary (("==" | "!=" | "<=" | ">=" | "<" | ">") as op,lhs,rhs) ->
        let instructions = Map.ofList ["==","if_icmpeq"; "!=","if_icmpne"; "<=","if_icmple"; ">=","if_icmpge"; "<","if_icmplt"; ">","if_icmpgt"]
        let left, leftSize = compileExpr context lhs
        let right, rightSize = compileExpr context rhs
        let yes, doneLabel = context.NewLabel(), context.NewLabel()
        left @ right @ [command (instructions.[op] + " " + yes); command "iconst_0"; command ("goto " + doneLabel); label yes; command "iconst_1"; label doneLabel], max leftSize (1 + rightSize)
    | Binary (("&&" | "||") as op,lhs,rhs) ->
        // Short circuit each operand, leaving exactly one normalized Boolean on the stack.
        let left, leftSize = compileExpr context lhs
        let right, rightSize = compileExpr context rhs
        let decisive, doneLabel = context.NewLabel(), context.NewLabel()
        let branch, defaultValue, decisiveValue = if op = "&&" then "ifeq", "iconst_1", "iconst_0" else "ifne", "iconst_0", "iconst_1"
        left @ [command (branch + " " + decisive)] @ right @ [command (branch + " " + decisive); command defaultValue; command ("goto " + doneLabel); label decisive; command decisiveValue; label doneLabel], max 1 (max leftSize rightSize)
    | Not value -> compileExpr context (Conditional(value,Number 0,Number 1))
    | Conditional (condition,yes,no) ->
        let test, testSize = compileExpr context condition
        let left, leftSize = compileExpr context yes
        let right, rightSize = compileExpr context no
        let otherwise, doneLabel = context.NewLabel(), context.NewLabel()
        test @ [command ("ifeq " + otherwise)] @ left @ [command ("goto " + doneLabel); label otherwise] @ right @ [label doneLabel], max testSize (max leftSize rightSize)
    | Call ("println",_) -> failwith "println returns void and is only valid as a statement"
    | Call (name,arguments) ->
        match Map.tryFind name context.Arities with
        | Some arity when arity = arguments.Length -> ()
        | Some arity -> failwithf "%s expects %d arguments, got %d" name arity arguments.Length
        | None -> failwithf "Unknown function: %s" name
        let instructions, size, _ =
            arguments |> List.fold (fun (commands,size,count) arg ->
                let generated, argSize = compileExpr context arg
                commands @ generated, max size (count + argSize), count + 1) ([],0,0)
        instructions @ [command (sprintf "invokestatic %s/%s(%s)I" context.ClassName name (String.replicate arguments.Length "I"))], max 1 size
    | Binary (op,_,_) -> failwithf "Unsupported operator: %s" op

The following is a test of the function `compile_expr`.

In [ ]:
let exampleContext = { Symbols=Map.ofList ["x",0; "y",1]; Arities=Map.ofList ["sum",1]; ClassName="Sum"; NewLabel=createLabels (); ReturnsVoid=false }
compileExpr exampleContext (Call("sum",[Binary("+",Variable "x",Variable "y")]))

`createLabels` returns a label factory with a private counter. Each method gets its own factory, preventing notebook execution history from changing generated labels.

In [ ]:
let newLabel = createLabels ()
newLabel (), newLabel ()

`compileExpr`, `compileBool` and `compileStatement` receive a `CompilationContext` and a typed AST. The context holds local-variable slots, function arities, class name and a label factory. Each returns `(instructions, maximumStack)`, so the JVM stack limit is computed while generating code. Arithmetic operands contribute different temporary stack heights; branch arms meet at equal heights.

In [ ]:
// Boolean expressions are normalized by the same typed expression compiler.
let compileBool context expression = compileExpr context expression

Below is a test for the function `compile_bool`.

In [ ]:
compileBool exampleContext (Binary("==",Variable "x",Number 0))

`compileExpr`, `compileBool` and `compileStatement` receive a `CompilationContext` and a typed AST. The context holds local-variable slots, function arities, class name and a label factory. Each returns `(instructions, maximumStack)`, so the JVM stack limit is computed while generating code. Arithmetic operands contribute different temporary stack heights; branch arms meet at equal heights.

In [ ]:
let rec compileStatement context statement : string list * int =
    let expr = compileExpr context
    let stmnt = compileStatement context
    match statement with
    | Assign (name,value) ->
        let commands,size = expr value
        match Map.tryFind name context.Symbols with
        | Some slot -> commands @ [indent ("istore " + string slot)], size
        | None -> failwithf "Assignment to undeclared variable: %s" name
    | If (condition,yes,no) ->
        let test,testSize = expr condition
        let yesCommands,yesSize = stmnt yes
        let noCommands,noSize = match no with Some other -> stmnt other | None -> [],0
        let otherwise,doneLabel = context.NewLabel(),context.NewLabel()
        test @ [indent ("ifeq " + otherwise)] @ yesCommands @ [indent ("goto " + doneLabel); label otherwise] @ noCommands @ [label doneLabel], max testSize (max yesSize noSize)
    | While (condition,body) ->
        let test,testSize = expr condition
        let bodyCommands,bodySize = stmnt body
        let loop,doneLabel = context.NewLabel(),context.NewLabel()
        [label loop] @ test @ [indent ("ifeq " + doneLabel)] @ bodyCommands @ [indent ("goto " + loop); label doneLabel], max testSize bodySize
    | Return value ->
        let commands,size = expr value
        commands @ (if context.ReturnsVoid then [indent "pop"; indent "return"] else [indent "ireturn"]), size
    | Block statements ->
        statements |> List.fold (fun (commands,size) statement -> let next,nextSize = stmnt statement in commands @ next, max size nextSize) ([],0)
    | Expression (Call("println",[value])) ->
        let commands,size = expr value
        [indent "getstatic java/lang/System/out Ljava/io/PrintStream;"] @ commands @ [indent "invokevirtual java/io/PrintStream/println(I)V"], 1 + size
    | Expression (Call("println",_)) -> failwith "println requires exactly one integer argument"
    | Expression value ->
        let commands,size = expr value
        commands @ [indent "pop"], size // Discard unused values to keep loop stack heights consistent.

In [ ]:
compileStatement exampleContext (If(Binary("==",Variable "x",Variable "y"),Assign("x",Number 42),None))

In [ ]:
compileStatement exampleContext (If(Binary("<",Variable "x",Variable "y"),Assign("x",Variable "y"),Some(Assign("y",Variable "x"))))

In [ ]:
compileStatement exampleContext (While(Binary("<",Variable "x",Variable "y"),Assign("x",Binary("+",Variable "x",Number 1))))

In [ ]:
compileStatement exampleContext (Block[Assign("x",Variable "y"); Block[Assign("x",Number 1); Assign("y",Variable "x")]])

In [ ]:
let compileFunction className arities definition =
    let isMain = definition.Name = "main"
    if isMain && not definition.Parameters.IsEmpty then failwith "main must have no C parameters"
    let names = definition.Parameters @ definition.Variables
    if (Set.ofList names).Count <> names.Length then failwithf "Duplicate local or parameter in %s" definition.Name
    // JVM main slot 0 contains String[] args, so Integer C locals start at slot 1.
    let firstSlot = if isMain then 1 else 0
    let symbols = names |> List.mapi (fun i name -> name,i+firstSlot) |> Map.ofList
    let context = {Symbols=symbols; Arities=arities; ClassName=className; NewLabel=createLabels (); ReturnsVoid=isMain}
    let commands,size = compileStatement context (Block definition.Body)
    let initialize = definition.Variables |> List.collect (fun name -> [indent "iconst_0"; indent ("istore " + string symbols.[name])])
    let signature = if isMain then ".method public static main([Ljava/lang/String;)V" else sprintf ".method public static %s(%s)I" definition.Name (String.replicate definition.Parameters.Length "I")
    let fallback = if isMain then [indent "return"] else [indent "iconst_0"; indent "ireturn"]
    [signature; sprintf ".limit locals %d" (firstSlot + names.Length); sprintf ".limit stack %d" (max 1 size)] @ initialize @ commands @ fallback @ [".end method"]

In [ ]:
let sumFunction = {Name="sum"; Parameters=["x"]; Variables=["y";"z"]; Body=[Return(Variable "x")]}
compileFunction "Sum" (Map.ofList ["sum",1]) sumFunction

In [ ]:
let file = "Examples/Primes.c"
printfn "%s" (Path.GetDirectoryName file)
printfn "%s" (Path.GetFileName file)
printfn "%s" (Path.Combine("abc","xyz.c"))

`compileProgram fileName` parses a `.c` file, checks function/local declarations and call arities, and writes sibling `.jas` assembly. JVM `main` reserves slot 0 for its `String[]` argument. Expression statements discard unused values and `println` is explicitly void.

In [ ]:
let compileProgram (fileName: string) =
    if Path.GetExtension(fileName) <> ".c" then invalidArg "fileName" "Expected a .c source file"
    let className = Path.GetFileNameWithoutExtension fileName
    if not (System.Text.RegularExpressions.Regex.IsMatch(className,@"^[A-Za-z][A-Za-z0-9_]*$")) then invalidArg "fileName" "Class name must be a JVM identifier"
    let functions = Frontend.parse (File.ReadAllText fileName)
    let functionNames = functions |> List.map (fun f -> f.Name)
    if (Set.ofList functionNames).Count <> functions.Length then failwith "Duplicate function definition"
    if not (List.contains "main" functionNames) then failwith "Program needs a main function"
    let arities = functions |> List.filter (fun f -> f.Name <> "main") |> List.map (fun f -> f.Name,f.Parameters.Length) |> Map.ofList
    let header = [".class public " + className; ".super java/lang/Object"; ""; ".method public <init>()V"; ".limit stack 1"; ".limit locals 1"; indent "aload_0"; indent "invokespecial java/lang/Object/<init>()V"; indent "return"; ".end method"; ""]
    let methods = functions |> List.collect (fun f -> compileFunction className arities f @ [""])
    let output = Path.ChangeExtension(fileName,".jas")
    File.WriteAllLines(output,header @ methods)
    output

The file `Primes.c` is a simple `C` program that computes the prime numbers in a naive way.

In [ ]:
Directory.GetFiles("Examples","*.c") |> Array.iter (printfn "%s")

In [ ]:
File.ReadAllText("Examples/Primes.c") |> printfn "%s"

In [ ]:
let primeAssembler = compileProgram "Examples/Primes.c"

In [ ]:
Directory.GetFiles("Examples","*.jas") |> Array.iter (printfn "%s")

In [ ]:
File.ReadAllText(primeAssembler) |> printfn "%s"

Optionally assemble the generated `.jas` file with Jasmin after installing a JDK and Jasmin. The notebook defaults to deterministic F# compilation and does not download or install external executables.

In [ ]:
open System.Diagnostics
let runTool (executable: string) (arguments: string list) (workingDirectory: string) =
    let info = ProcessStartInfo(executable)
    info.WorkingDirectory <- workingDirectory
    info.UseShellExecute <- false
    info.RedirectStandardOutput <- true
    info.RedirectStandardError <- true
    for argument in arguments do info.ArgumentList.Add argument
    use childProcess = Process.Start(info)
    let output = childProcess.StandardOutput.ReadToEnd()
    let errors = childProcess.StandardError.ReadToEnd()
    childProcess.WaitForExit()
    if childProcess.ExitCode <> 0 then failwithf "%s failed: %s" executable errors
    output
// Optional external execution after installing a JDK and Jasmin:
// runTool "jasmin" ["Primes.jas"] "Examples" |> printfn "%s"

After assembly, run the class with Java. This step requires the external tools; successful F# execution alone does not prove JVM execution.

In [ ]:
// Generated source files can be inspected before optional JVM assembly.
for file in Directory.GetFiles("Examples","*.c") do
    let assembler = compileProgram file
    let text = File.ReadAllText assembler
    checkNotebook (text.Contains(".limit stack") && text.Contains(".limit locals") && text.Contains("main([Ljava/lang/String;)V"))
    printfn "Compiled %s" file

In [ ]:
// Optional after assembly: runTool "java" ["-cp"; "."; "Primes"] "Examples" |> printfn "%s"

## Verification

These checks cover arithmetic and Boolean precedence, dangling else, malformed input, typed symbol boundaries and JVM stack balance. The optional JVM script described in README.md assembles and executes all six bundled examples with bytecode verification enabled. It also checks that `!x < y` and `!(x < y)` both produce zero for `x = 1`, `y = 2`, and that `&&`/`||` skip a right operand containing division by zero or a function call with observable output.


In [ ]:
let expectFailure action =
    let mutable failed = false
    try action () with _ -> failed <- true
    checkNotebook failed
expectFailure (fun () -> Frontend.parse "int main() { @ }" |> ignore)
expectFailure (fun () -> Frontend.parse "int main() { println(1);" |> ignore)
let precedenceAst = Frontend.parse "int main() { println(1 + 2 * 3); }"
match precedenceAst.Head.Body with
| [Expression(Call("println",[Binary("+",Number 1,Binary("*",Number 2,Number 3))]))] -> ()
| _ -> failwith "Incorrect arithmetic precedence"
// The lecture places Boolean negation below comparison operators.
let negatedComparison = Not(Binary("<",Variable "x",Variable "y"))
for source in ["!x < y"; "!(x < y)"] do
    let ast = Frontend.parse ("int main() { println(" + source + "); }")
    match ast.Head.Body with
    | [Expression(Call("println",[value]))] when value = negatedComparison -> ()
    | _ -> failwithf "Incorrect Boolean negation precedence: %s" source
let danglingElse = Frontend.parse "int main() { if (1) if (0) println(1); else println(2); }"
match danglingElse.Head.Body with
| [If(_,If(_,_,Some _),None)] -> ()
| _ -> failwith "Else must attach to the inner if"
expectFailure (fun () -> compileExpr exampleContext (Variable "undeclared") |> ignore)
expectFailure (fun () -> compileExpr exampleContext (Call("sum",[])) |> ignore)
let unusedValue,_ = compileStatement exampleContext (Expression(Number 42))
checkNotebook (unusedValue |> List.last = indent "pop")
let booleanCommands,booleanStack = compileBool exampleContext (Not(Binary("==",Variable "x",Number 0)))
checkNotebook (booleanStack >= 2 && booleanCommands |> List.exists (fun line -> line.Contains("ifeq")))
printfn "Arithmetic/Boolean precedence, diagnostics, arity, unused values and Boolean code generation passed."
